# Crop Yield Risk Forecasting

This notebook addresses three specific points:

1. **Reframe as forecasting, not circular prediction** — predictors are computed from an earlier window of years; the risk label is computed from a separate, later window of years, so the model is never given information that was used to build its own target.
2. **Expand the sample size** using rolling 3-year windows instead of one row per state-crop pair.
3. **Add climate-variability features** so rainfall has a genuine chance to contribute alongside yield history and soil.

## 1. Load the dataset

In [9]:
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

df = pd.read_csv(r'C:\MCA RESEARCH\agri_ai_project\data\final\agri_combined_datasets.csv')
df['annual_rainfall'] = pd.to_numeric(df['annual_rainfall'], errors='coerce')
df.shape

(6817, 25)

## 2. Collapse district-level rows into one row per state-crop-year

In [10]:
soil_cols = ['n_high','n_medium','n_low','p_high','p_medium','p_low','k_high','k_medium','k_low',
             'oc_high','oc_medium','oc_low','ph_alkaline','ph_acidic','ph_neutral','ec_nonsaline','ec_saline']

def agg_group(g):
    area_sum = g['area'].sum()
    prod_sum = g['production'].sum()
    yield_weighted = prod_sum/area_sum if area_sum > 0 else g['yield'].mean()
    out = {'yield': yield_weighted, 'annual_rainfall': g['annual_rainfall'].mean()}
    for c in soil_cols:
        out[c] = g[c].mean()
    return pd.Series(out)

panel = df.groupby(['state','crop','year']).apply(agg_group, include_groups=False).reset_index()
print(panel.shape)
panel.groupby(['state','crop'])['year'].count().describe()

(275, 22)


count    25.0
mean     11.0
std       0.0
min      11.0
25%      11.0
50%      11.0
75%      11.0
max      11.0
Name: year, dtype: float64

## 3. Reframe as forecasting + expand the sample with rolling windows

For every state-crop pair, slide a window across its years: the first 3 years become the **predictors**, and the *following, non-overlapping* 3 years become the **target**. This guarantees the label is never computed from the same years as the predictors, and turns each pair's single timeline into multiple training rows instead of one.

In [11]:
F, T = 3, 3  # 3 years of features, followed by 3 years used to build the target
rows = []
for (state, crop), g in panel.groupby(['state','crop']):
    g = g.sort_values('year').reset_index(drop=True)
    years = g['year'].tolist()
    ymin, ymax = min(years), max(years)
    for s in range(ymin, ymax - (F+T) + 2):
        feat_years = list(range(s, s+F))
        targ_years = list(range(s+F, s+F+T))
        if not all(y in years for y in feat_years + targ_years):
            continue
        fw = g[g['year'].isin(feat_years)]
        tw = g[g['year'].isin(targ_years)]

        feat_yield_mean = fw['yield'].mean()
        feat_yield_cv = fw['yield'].std()/feat_yield_mean if feat_yield_mean else np.nan

        # climate-variability features: mean rainfall and rainfall CV over the feature window
        feat_rain_mean = fw['annual_rainfall'].mean()
        feat_rain_cv = fw['annual_rainfall'].std()/feat_rain_mean if feat_rain_mean else np.nan

        targ_yield_mean = tw['yield'].mean()
        targ_yield_cv = tw['yield'].std()/targ_yield_mean if targ_yield_mean else np.nan

        row = {'state':state, 'crop':crop, 'feat_start':s, 'feat_end':s+F-1,
               'targ_start':s+F, 'targ_end':s+F+T-1,
               'feat_yield_mean':feat_yield_mean, 'feat_yield_cv':feat_yield_cv,
               'feat_rain_mean':feat_rain_mean, 'feat_rain_cv':feat_rain_cv,
               'target_cv':targ_yield_cv}
        for c in soil_cols:
            row[c] = fw[c].mean()
        rows.append(row)

wdf = pd.DataFrame(rows)
print(wdf.shape)
wdf.groupby(['state','crop']).size().describe()

(150, 28)


count    25.0
mean      6.0
std       0.0
min       6.0
25%       6.0
50%       6.0
75%       6.0
max       6.0
dtype: float64

### All 150 rolling-window rows

Every row here is one (state, crop, 3-year feature window) combination. feat_start/feat_end mark the years used for predictors;  targ_start/targ_end mark the separate, later years used to build target_cv.

In [12]:
print(f"Total rows: {len(wdf)}\n")
wdf.head(10) 

Total rows: 150



,state,crop,feat_start,feat_end,targ_start,targ_end,feat_yield_mean,feat_yield_cv,feat_rain_mean,feat_rain_cv,...,k_medium,k_low,oc_high,oc_medium,oc_low,ph_alkaline,ph_acidic,ph_neutral,ec_nonsaline,ec_saline
0,andhra pradesh,Cotton(lint),2012,2014,2015,2017,3.070789,0.129866,1002.941026,0.148121,...,30.437692,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462
1,andhra pradesh,Cotton(lint),2013,2015,2016,2018,3.109806,0.106786,948.292308,0.108712,...,30.437692,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462
2,andhra pradesh,Cotton(lint),2014,2016,2017,2019,3.140650,0.112665,911.992308,0.071648,...,30.437692,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462
3,andhra pradesh,Cotton(lint),2015,2017,2018,2020,3.100335,0.103981,953.874359,0.017978,...,30.437692,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462
4,andhra pradesh,Cotton(lint),2016,2018,2019,2021,2.990891,0.170713,927.284615,0.043843,...,30.437692,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462
5,andhra pradesh,Cotton(lint),2017,2019,2020,2022,2.096340,0.629940,954.702564,0.070005,...,30.437692,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462
6,andhra pradesh,Groundnut,2012,2014,2015,2017,0.520996,0.208492,994.109829,0.138680,...,29.680128,19.704017,21.886538,28.402991,49.671731,4.421795,0.283974,95.243632,94.087137,5.912863
7,andhra pradesh,Groundnut,2013,2015,2016,2018,0.637707,0.366050,948.292308,0.108712,...,30.437692,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462
8,andhra pradesh,Groundnut,2014,2016,2017,2019,0.568297,0.469501,911.992308,0.071648,...,30.437692,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462
9,andhra pradesh,Groundnut,2015,2017,2018,2020,0.862051,0.506225,953.874359,0.017978,...,30.437692,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462


## 4. Time-based train/test split, then bin the target into Low/Medium/High

In [ ]:
train = wdf[wdf['feat_start'].isin([2012,2013,2014])].copy()
test  = wdf[wdf['feat_start'].isin([2015,2016,2017])].copy()

q33, q67 = train['target_cv'].quantile([0.33, 0.67])
def to_label(cv):
    if cv < q33: return 'Low'
    elif cv < q67: return 'Medium'
    else: return 'High'

train['risk'] = train['target_cv'].apply(to_label)
test['risk']  = test['target_cv'].apply(to_label)
print('thresholds fit on train only:', round(q33,3), round(q67,3))
print(train['risk'].value_counts())
print(test['risk'].value_counts())

Risk thresholds fitted using TRAINING data only:
33rd percentile = 0.124
67th percentile = 0.341

Training samples: 50
risk
High      18
Low       16
Medium    16
Name: count, dtype: int64

Testing samples: 25
risk
Low       12
Medium     7
High       6
Name: count, dtype: int64


In [18]:
# ==========================================================
# FINAL FUTURE HOLDOUT: 2019-2022
# ==========================================================

future_rows = []

for (state, crop), g in panel.groupby(["state", "crop"]):

    g = g.sort_values("year")

    # Historical feature period
    fw = g[g["year"].between(2016, 2018)]

    # Completely unseen future target period
    tw = g[g["year"].between(2019, 2022)]

    # Require complete historical and future periods
    if len(fw) != 3 or len(tw) != 4:
        continue

    # ---------------- FEATURE PERIOD ----------------

    feat_yield_mean = fw["yield"].mean()

    feat_yield_cv = (
        fw["yield"].std() / feat_yield_mean
        if feat_yield_mean != 0
        else np.nan
    )

    feat_rain_mean = fw["annual_rainfall"].mean()

    feat_rain_std = fw["annual_rainfall"].std()

    feat_rain_cv = (
        feat_rain_std / feat_rain_mean
        if feat_rain_mean != 0
        else np.nan
    )

    # ---------------- FUTURE TARGET ----------------

    future_yield_mean = tw["yield"].mean()

    future_cv = (
        tw["yield"].std() / future_yield_mean
        if future_yield_mean != 0
        else np.nan
    )

    row = {
        "state": state,
        "crop": crop,

        "feat_start": 2016,
        "feat_end": 2018,

        "targ_start": 2019,
        "targ_end": 2022,

        "feat_yield_mean": feat_yield_mean,
        "feat_yield_cv": feat_yield_cv,

        "feat_rain_mean": feat_rain_mean,
        "feat_rain_std": feat_rain_std,
        "feat_rain_cv": feat_rain_cv,

        "target_cv": future_cv
    }

    # Soil is treated as contextual/static
    for col in soil_cols:
        row[col] = fw[col].mean()

    future_test = row
    future_rows.append(row)


future_test = pd.DataFrame(future_rows)

print("Future holdout samples:", len(future_test))
future_test.head()

Future holdout samples: 25


,state,crop,feat_start,feat_end,targ_start,targ_end,feat_yield_mean,feat_yield_cv,feat_rain_mean,feat_rain_std,...,k_medium,k_low,oc_high,oc_medium,oc_low,ph_alkaline,ph_acidic,ph_neutral,ec_nonsaline,ec_saline
0,andhra pradesh,Cotton(lint),2016,2018,2019,2022,2.990891,0.170713,927.284615,40.654728,...,30.437692,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462
1,andhra pradesh,Groundnut,2016,2018,2019,2022,0.731645,0.663786,927.284615,40.654728,...,30.437692,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462
2,andhra pradesh,Maize,2016,2018,2019,2022,4.320219,0.071675,927.284615,40.654728,...,30.437692,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462
3,andhra pradesh,Ragi,2016,2018,2019,2022,1.149371,0.155813,957.675758,34.591656,...,34.900909,16.434545,22.676364,27.318182,49.960909,4.603636,0.326364,95.010909,93.479091,6.520909
4,andhra pradesh,Rice,2016,2018,2019,2022,3.380930,0.016184,927.284615,40.654728,...,30.437692,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462


### Training and testing rows

**Train** = rows whose feature window starts in 2012, 2013, or 2014.
**Test** = rows whose feature window starts in 2015, 2016, or 2017 (a later, non-overlapping period).

In [14]:
print(f"Train: {len(train)} rows")
display(train.head(10))
print(f"\nTest: {len(test)} rows")
display(test.head(10))

Train: 75 rows


,state,crop,feat_start,feat_end,targ_start,targ_end,feat_yield_mean,feat_yield_cv,feat_rain_mean,feat_rain_cv,...,k_low,oc_high,oc_medium,oc_low,ph_alkaline,ph_acidic,ph_neutral,ec_nonsaline,ec_saline,risk
0,andhra pradesh,Cotton(lint),2012,2014,2015,2017,3.070789,0.129866,1002.941026,0.148121,...,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462,Low
1,andhra pradesh,Cotton(lint),2013,2015,2016,2018,3.109806,0.106786,948.292308,0.108712,...,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462,Medium
2,andhra pradesh,Cotton(lint),2014,2016,2017,2019,3.140650,0.112665,911.992308,0.071648,...,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462,High
6,andhra pradesh,Groundnut,2012,2014,2015,2017,0.520996,0.208492,994.109829,0.138680,...,19.704017,21.886538,28.402991,49.671731,4.421795,0.283974,95.243632,94.087137,5.912863,High
7,andhra pradesh,Groundnut,2013,2015,2016,2018,0.637707,0.366050,948.292308,0.108712,...,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462,High
8,andhra pradesh,Groundnut,2014,2016,2017,2019,0.568297,0.469501,911.992308,0.071648,...,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462,High
12,andhra pradesh,Maize,2012,2014,2015,2017,3.697057,0.107680,1002.941026,0.148121,...,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462,Medium
13,andhra pradesh,Maize,2013,2015,2016,2018,3.561719,0.065688,948.292308,0.108712,...,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462,Low
14,andhra pradesh,Maize,2014,2016,2017,2019,3.707742,0.115746,911.992308,0.071648,...,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462,Low
18,andhra pradesh,Ragi,2012,2014,2015,2017,1.020430,0.052830,1023.288258,0.132116,...,16.178864,20.552992,28.502121,50.914773,4.633258,0.315909,95.011856,94.035227,5.964773,Medium



Test: 75 rows


,state,crop,feat_start,feat_end,targ_start,targ_end,feat_yield_mean,feat_yield_cv,feat_rain_mean,feat_rain_cv,...,k_low,oc_high,oc_medium,oc_low,ph_alkaline,ph_acidic,ph_neutral,ec_nonsaline,ec_saline,risk
3,andhra pradesh,Cotton(lint),2015,2017,2018,2020,3.100335,0.103981,953.874359,0.017978,...,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462,High
4,andhra pradesh,Cotton(lint),2016,2018,2019,2021,2.990891,0.170713,927.284615,0.043843,...,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462,High
5,andhra pradesh,Cotton(lint),2017,2019,2020,2022,2.096340,0.629940,954.702564,0.070005,...,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462,Low
9,andhra pradesh,Groundnut,2015,2017,2018,2020,0.862051,0.506225,953.874359,0.017978,...,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462,High
10,andhra pradesh,Groundnut,2016,2018,2019,2021,0.731645,0.663786,927.284615,0.043843,...,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462,High
11,andhra pradesh,Groundnut,2017,2019,2020,2022,0.957131,0.439172,954.702564,0.070005,...,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462,Medium
15,andhra pradesh,Maize,2015,2017,2018,2020,4.169600,0.119483,953.874359,0.017978,...,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462,Low
16,andhra pradesh,Maize,2016,2018,2019,2021,4.320219,0.071675,927.284615,0.043843,...,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462,Medium
17,andhra pradesh,Maize,2017,2019,2020,2022,4.509575,0.072307,954.702564,0.070005,...,20.157692,21.792308,28.546154,49.623846,4.307692,0.278462,95.364615,94.221538,5.778462,Low
21,andhra pradesh,Ragi,2015,2017,2018,2020,1.061577,0.127692,986.403030,0.027759,...,16.434545,22.676364,27.318182,49.960909,4.603636,0.326364,95.010909,93.479091,6.520909,Low


## 5. Train and compare models

Compare a yield-only baseline against a model that includes rainfall and soil, to check whether the climate-variability features add anything.

In [15]:
feat_sets = {
 'Baseline (mean yield only)': ['feat_yield_mean'],
 'Mean + past-window CV': ['feat_yield_mean','feat_yield_cv'],
 'Mean + CV + rainfall + soil': ['feat_yield_mean','feat_yield_cv','feat_rain_mean','feat_rain_cv'] + soil_cols,
}
results = {}
for name, cols in feat_sets.items():
    X_tr, y_tr = train[cols].fillna(0), train['risk']
    X_te, y_te = test[cols].fillna(0), test['risk']
    clf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
    clf.fit(X_tr, y_tr)
    pred = clf.predict(X_te)
    acc, f1 = accuracy_score(y_te, pred), f1_score(y_te, pred, average='weighted')
    results[name] = (acc, f1, clf, pred)
    print(f"{name}: accuracy={acc:.3f}  F1={f1:.3f}")

# Best model = highest weighted F1 on the test set (F1 balances precision/recall
# across the three risk classes better than accuracy alone, since the classes
# aren't perfectly balanced)
best_name = max(results, key=lambda n: results[n][1])
print(f"\nBest model: '{best_name}' (F1={results[best_name][1]:.3f}, accuracy={results[best_name][0]:.3f})")

Baseline (mean yield only): accuracy=0.440  F1=0.448
Mean + past-window CV: accuracy=0.440  F1=0.449
Mean + CV + rainfall + soil: accuracy=0.507  F1=0.529

Best model: 'Mean + CV + rainfall + soil' (F1=0.529, accuracy=0.507)


## 6. Confusion matrix + feature importance for the best model


In [16]:
acc, f1, clf, pred = results[best_name]
print(f"Using best model: '{best_name}'\n")

labels = ['Low','Medium','High']
cm = confusion_matrix(test['risk'], pred, labels=labels)
print(pd.DataFrame(cm, index=labels, columns=labels))
print(classification_report(test['risk'], pred, labels=labels, zero_division=0))

feat_cols = feat_sets[best_name]
imp = pd.Series(clf.feature_importances_, index=feat_cols).sort_values(ascending=False)
imp.head(10)

Using best model: 'Mean + CV + rainfall + soil'

        Low  Medium  High
Low      21       5    13
Medium    4       8    10
High      1       4     9
              precision    recall  f1-score   support

         Low       0.81      0.54      0.65        39
      Medium       0.47      0.36      0.41        22
        High       0.28      0.64      0.39        14

    accuracy                           0.51        75
   macro avg       0.52      0.51      0.48        75
weighted avg       0.61      0.51      0.53        75



feat_yield_cv      0.131538
feat_yield_mean    0.124226
feat_rain_mean     0.072391
feat_rain_cv       0.063479
p_low              0.060071
oc_medium          0.052294
n_medium           0.050060
n_low              0.049780
p_high             0.038142
ph_acidic          0.036928
dtype: float64